In [1]:
import glob
from lxml import etree
from pprint import pprint
from pydantic import BaseModel

from collections import Counter, defaultdict
from tqdm import tqdm
from multiprocessing import Pool

import pubmed_parser as pp


In [4]:
filepaths = sorted(list(glob.glob("../data/raw/oa_comm/PMC*/PMC*.xml")))
print(len(filepaths))

2661122


In [5]:
articles = []
bad_filepaths = []
with Pool(10) as p:
    for x in tqdm(p.imap(pp.pmc_parser.extract_article, filepaths[:100_000]), total=len(filepaths[:100_000])):
        if isinstance(x, str):
            bad_filepaths.append(x)
        else:
            articles.append(x)

100%|██████████| 100000/100000 [01:59<00:00, 833.45it/s]


In [6]:
sum(bool(article.pub_date) for article in articles)

100000

In [7]:
import json
for article in articles:
    out_path = f"../data/extracted/oa_comm/{article.article_ids.pmc}.json"
    json.dump(article.model_dump(), open(out_path, 'w'))

FileNotFoundError: [Errno 2] No such file or directory: '../data/extracted/oa_comm/PMC176545.json'

In [11]:
import random
import json

for i in range(5):
    idx = random.randint(0, len(articles))
    print(json.dumps(articles[idx].model_dump()))

{"schema_version": "1.0.0", "article_type": "research-article", "article_ids": {"pmc": "PMC2395481", "pmid": null, "doi": "10.1080/135771402320883862", "publisher_id": null, "pii": "S1357714X02000233"}, "pub_date": {"year": "2002", "month": "09", "day": null}, "subjects": ["Research Article"], "title": "Oral Presentations", "subtitle": null, "abstract": null, "references": []}
{"schema_version": "1.0.0", "article_type": "research-article", "article_ids": {"pmc": "PMC2749028", "pmid": "19723333", "doi": "10.1186/1471-2369-10-24", "publisher_id": "1471-2369-10-24"}, "pub_date": {"year": "2009", "month": "09", "day": "01"}, "subjects": ["Research Article"], "title": "Estimated GFR reporting is not sufficient to allow detection of chronic kidney disease in an Italian regional hospital", "subtitle": null, "abstract": "Background Chronic kidney disease (CKD) is an emerging worldwide problem. The lack of attention paid to kidney disease is well known and has been described in previous publica